# KisanAI - Market Data Analysis

KisanAI / Sampoorn Kisan AI Sahayak

## Goal
Explore AGMARKET data and build a baseline market-price analysis pipeline.
This notebook first identifies the actual columns in your downloaded dataset before making assumptions.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
from google.colab import files
uploaded = files.upload()

import glob, pandas as pd, os
files_found = glob.glob("/content/**/*.csv", recursive=True)
print(files_found)

# Select the relevant CSV after inspecting the list
CSV_PATH = files_found[0]
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df.describe(include="all").T.head(30))

In [ ]:
# Basic cleanup and missing-value report
print(df.isnull().sum().sort_values(ascending=False).head(30))

# Show likely categorical/date columns
for c in df.columns:
    if df[c].dtype == "object":
        print(c, "->", df[c].dropna().astype(str).head(5).tolist())

## Identify numeric price columns

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Numeric columns:", numeric_cols)

# Choose a likely price column manually after seeing the output.
# Example:
# PRICE_COL = "Modal Price"
PRICE_COL = numeric_cols[-1] if numeric_cols else None
print("Initial price column choice:", PRICE_COL)

In [ ]:
if PRICE_COL:
    import matplotlib.pyplot as plt
    df[PRICE_COL].plot(kind="hist", bins=40)
    plt.title("Price distribution")
    plt.xlabel(PRICE_COL)
    plt.show()

    print(df[PRICE_COL].describe())

## Optional baseline forecasting
If the dataset contains a date column and one numeric price column, this creates a simple time-based baseline.
It is not a production forecasting model.

In [ ]:
date_candidates = []
for c in df.columns:
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() > 0.6:
        date_candidates.append(c)

print("Possible date columns:", date_candidates)

In [ ]:
# Set these manually after inspecting the previous output.
# DATE_COL = "Date"
# PRICE_COL = "Modal Price"

if 'DATE_COL' in globals() and DATE_COL in df.columns and PRICE_COL in df.columns:
    tmp = df[[DATE_COL, PRICE_COL]].copy()
    tmp[DATE_COL] = pd.to_datetime(tmp[DATE_COL], errors="coerce")
    tmp = tmp.dropna().sort_values(DATE_COL)

    daily = tmp.groupby(DATE_COL)[PRICE_COL].mean().reset_index()
    daily.plot(x=DATE_COL, y=PRICE_COL, figsize=(12,5), title="Average market price over time")
    plt.show()
else:
    print("Set DATE_COL and PRICE_COL after inspecting your dataset.")